# Silver: extracción de Pose World Landmarks

Procesa los JPEG de `data/silver/extracted_frames/<exercise>/<video_id>/` con MediaPipe Pose Landmarker en modo `VIDEO`. Guarda un Parquet por video en `data/silver/landmarks/<exercise>/` y un Parquet de control de calidad por ejercicio.

Se conserva una fila por frame, incluidos los frames sin pose. Los timestamps se reconstruyen en milisegundos a partir del índice del frame fuente, los FPS originales del video y el muestreo determinista de 7 FPS usado por `01_frames_extractor.ipynb`. Esta reconstrucción asume video de frecuencia constante (CFR); no se cambian ni se sobrescriben videos o JPEG.

Antes de ejecutar, instala `requirements.txt` y coloca el archivo oficial Lite `pose_landmarker.task` en `models/pose/`. Descarga: [Pose Landmarker Lite](https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task).

In [ ]:
import logging
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
MODEL_PATH = PROJECT_ROOT / 'models' / 'pose' / 'pose_landmarker.task'

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)s: %(message)s',
)
print(f'Raíz del proyecto: {PROJECT_ROOT}')
print(f'Modelo Pose Landmarker: {MODEL_PATH}')

## Ejecutar extracción

El pipeline valida la correspondencia entre metadata, videos y carpetas de frames. Cada Parquet contiene las 33 landmarks y los cinco atributos soportados por el SDK (`x`, `y`, `z`, `visibility`, `presence`); los atributos no disponibles se guardan como nulos. Los resultados existentes no se reemplazan salvo que se indique explícitamente `overwrite=True`.

In [ ]:
from src.pose.landmarks import extract_exercise_landmarks

landmark_results = []
for exercise in ('plank', 'hollow_body'):
    result = extract_exercise_landmarks(
        PROJECT_ROOT,
        exercise,
        MODEL_PATH,
        sampled_fps=7.0,
        overwrite=False,
    )
    landmark_results.append(result)
landmark_results